In [ ]:
import re
import os
import ssl
import arxiv
import certifi
import requests
import itertools
import urllib.request
from scholarly import scholarly
from langchain.tools import tool
from langchain_groq import ChatGroq
from semanticscholar import SemanticScholar
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from typing import TypedDict, List,Optional, Annotated,Sequence,Set, Dict
from langchain_core.messages import BaseMessage,SystemMessage,HumanMessage

In [ ]:
class Clarification(TypedDict):
    needs_improvement :  bool
    questions : Optional[List[str]]

class KeywordExtractionOutput(TypedDict):
    google_scholar_queries: List[str]

class Selection(TypedDict):
    paper_titles: List[str]

class AgentState(TypedDict):
    clarification : Clarification
    keywords : KeywordExtractionOutput
    messages : Annotated[Sequence[BaseMessage],add_messages]
    # arxiv_papers : Set[str]
    # ss_papers : Set[str]
    scholar_titles : Set[str]
    selected_papers_1 : List[str]
    selected_papers_2 : List[str]
    metadata: Dict[str,List[str]]

In [43]:
def sanitize_filename(name: str) -> str:
    title = re.sub(r'[\\/*?:"<>|]', "_", name)
    return title.replace(" ","_")

def download_pdf(url: str, filepath: str):
    resp = requests.get(url)
    resp.raise_for_status()
    with open(filepath, "wb") as f:
        f.write(resp.content)

def save_abstract(text: str, filepath: str):
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)

In [44]:
clarifier_llm = ChatGroq(model="moonshotai/kimi-k2-instruct").with_structured_output(Clarification)
keyword_llm = ChatGroq(model="llama3-70b-8192").with_structured_output(KeywordExtractionOutput)
Selecter_llm = ChatGroq(model="moonshotai/kimi-k2-instruct").with_structured_output(Selection)

In [45]:
def scholar_searcher(state: AgentState):
    """
    Given a list of keywords. it return 10 most revlevent papers for each keyword.
    """
    title_results = set()

    for query in state['keywords']['google_scholar_queries']:
        count = 0
        for pub in scholarly.search_pubs(query):
            title_results.add(pub["bib"]["title"])
            count += 1
            if count >= 10:
                break 
    
    return {"scholar_titles": title_results}

In [46]:
def clarifier(state:AgentState):
    system_prompt = SystemMessage(content="""
        You are a research assistant. The user will provide you with a research paper topic or description.
        
        Your job is to check if the description includes:
        1. Research domain
        2. Problem being solved
        3. Method or technique used
        4. Any dataset mentioned
        
        If ANY of these elements are missing or unclear, set needs_improvement to True and provide specific questions in the 'questions' field to gather the missing information.
        
        If all elements are present and clear, set needs_improvement to False and questions can be null or empty.
        
        Example response format:
        - If missing info: {"needs_improvement": true, "questions": ["What specific problem are you trying to solve?", "Which dataset will you use?"]}
        - If complete: {"needs_improvement": false, "questions": null}
        """
    )
    clarifier_response = clarifier_llm.invoke([system_prompt]+[state["messages"][-1]])

    return {"clarification":clarifier_response}

In [47]:
def keyworder(state:AgentState):
    system_prompt = SystemMessage(content="""
    You are a research assistant trained to extract keywords for academic paper searches, specifically for:
    - Google Scholar (https://scholar.google.com)

    You will receive a short research topic description from the user.

    Your task is to analyze the description and return a set of structured keywords and search phrases optimized for Google Scholar.

    You MUST return your output as an instance of the following schema:

    class KeywordExtractionOutput(BaseModel):
        scholar_queries: List[str]  # concise search phrases optimized for Google Scholar

    Guidelines:
    - Avoid generic terms like "paper", "study", "research"
    - Prefer specific methods (e.g., CNN, BERT, PCA), tasks (e.g., segmentation, prediction), and datasets (e.g., CHB-MIT, ImageNet)
    - If user input is vague, extract the most relevant, inferable terms — don't leave the list empty
    - All outputs should be lowercase unless referring to acronyms (e.g., EEG, GNN, LSTM)

    Only return a valid Python object matching the schema exactly.
    Do not include any extra fields, strings, comments, or explanations.
    Avoid quoting the entire object as a string.
    """
    )
    
    keyworder_response = keyword_llm.invoke([system_prompt] + state['messages'])

    return {"keywords":keyworder_response}
    

In [ ]:
def Selector_1(state: AgentState):
    description = state['messages'][-1].content
    title_list = state['scholar_titles']
    titles = title_list[len(title_list) // 2 :]

    prompt = SystemMessage(content=f"""
    You are an academic assistant helping researchers identify relevant papers for literature reviews.

    Paper description:
    {description}

    Candidate paper titles:
    {titles}

    Task:
    - Select only titles that are highly relevant and could realistically be cited.
    - Be strict: exclude anything unrelated or loosely connected.
    """)

    result = Selecter_llm.invoke([prompt])  # should return Selection TypedDict
    return {"selected_papers_1": result["paper_titles"]}


def Selector_2(state: AgentState):
    description = state['messages'][-1].content
    title_list = state['scholar_titles']
    titles = title_list[len(title_list) // 2 :]

    prompt = SystemMessage(content=f"""
    You are an academic assistant helping researchers identify relevant papers for literature reviews.

    Paper description:
    {description}

    Candidate paper titles:
    {titles}

    Task:
    - Select only titles that are highly relevant and could realistically be cited.
    - Be strict: exclude anything unrelated or loosely connected.
    """)

    result = Selecter_llm.invoke([prompt])
    return {"selected_papers_2": result["paper_titles"]}


In [ ]:
import requests
from scholarly import scholarly   # pip install scholarly

def metadata_getter(state: dict):
    """
    Query Crossref for each paper title. 
    If Crossref fails, fallback to Google Scholar (scholarly).
    
    Returns:
    { "metadata": { "title": [doi, journal_name, authors_string] } }
    """
    base_url = "https://api.crossref.org/works"
    headers = {"User-Agent": "PaperMetadataFetcher/1.0 (mailto:your-email@example.com)"}
    
    results = {}
    titles = state['selected_papers_1'] + state['selected_papers_2']
    
    for title in titles:
        params = {"query.title": title, "rows": 1}
        response = requests.get(base_url, params=params, headers=headers)
        
        doi, journal, authors_str = "N/A", "N/A", "N/A"
        
        if response.status_code == 200:
            items = response.json().get("message", {}).get("items", [])
            if items:
                item = items[0]
                doi = item.get("DOI", "N/A")
                journal = (
                    item.get("container-title", ["N/A"])[0]
                    if item.get("container-title")
                    else "N/A"
                )
                authors = []
                for a in item.get("author", []):
                    name_parts = []
                    if "given" in a:
                        name_parts.append(a["given"])
                    if "family" in a:
                        name_parts.append(a["family"])
                    authors.append(" ".join(name_parts))
                authors_str = ", ".join(authors) if authors else "N/A"
        
        # 🔄 Fallback to Google Scholar if Crossref failed
        if journal == "N/A" or authors_str == "N/A":
            try:
                search_query = scholarly.search_pubs(title)
                paper = next(search_query, None)
                if paper:
                    bib = paper.get("bib", {})
                    journal = bib.get("venue", journal)  # venue is journal/conference in scholarly
                    authors_str = ", ".join(bib.get("author", [])) if bib.get("author") else authors_str
            except Exception as e:
                print(f"Google Scholar fallback failed for '{title}': {e}")
        
        results[title] = [doi, journal, authors_str]
    
    return {"metadata": results}

In [49]:
def clarifier_router(state: AgentState):
    if state['clarification']['needs_improvement']:
        return "end"
    else:
        return "continue"

In [ ]:
def merge_results(state: AgentState):
    return {
        "selected_papers": state.get("selected_papers_1", []) 
                          + state.get("selected_papers_2", [])
    }

graph = StateGraph(AgentState)

graph.add_node("clarificationAgent", clarifier)
graph.add_node("keywordAgent", keyworder)
graph.add_node("scholar_searcher", scholar_searcher)
graph.add_node("selecter_1", Selector_1)
graph.add_node("selecter_2", Selector_2)
graph.add_node("merger", merge_results)
graph.add_node("metadata_getter", metadata_getter)

graph.add_edge(START, "clarificationAgent")

graph.add_conditional_edges(
    "clarificationAgent",
    clarifier_router,
    {
        "end": END,
        "continue": "keywordAgent"
    }
)

graph.add_edge("keywordAgent", "scholar_searcher")
graph.add_edge("scholar_searcher", "selecter_1")
graph.add_edge("scholar_searcher", "selecter_2")

# Both selectors flow into merger
graph.add_edge("selecter_1", "metadata_getter")
graph.add_edge("selecter_2", "metadata_getter")

graph.add_edge("metadata_getter", END)

app = graph.compile()


In [51]:
user_input_messages = [HumanMessage(content="""
The title of my paper is preictal state recognition using geometric deep learning. Im trying to improve the early detection of preictal (pre-seizure) brain states in patients with epilepsy using EEG data. 
                                    The goal is to predict seizure onset several minutes in advance so preventive interventions can be applied, especially in wearable or edge devices. 
                                    I plan to use a Graph Neural Network (GNN) architecture, specifically a spatio-temporal GCN, to model both the spatial brain connectivity and the temporal patterns leading up to a seizure
                                    .Ill be using the CHB-MIT Scalp EEG dataset, 
                                    which contains long-term EEG recordings from pediatric subjects with intractable seizures, 
                                    including annotations for seizure onset and preictal windows.
""")]

# Initialize state properly
state = {
    "messages": user_input_messages, 
    "clarification": {"needs_improvement": False, "question": None}, 
    "keywords": ""
}

for event in app.stream(state):
    for node_name, node_output in event.items():
        print(f"\n🧩 Agent: {node_name}")
        print(f"📦 Output: {node_output}")
        state.update(node_output)
        
    if node_name == END:
        break
        
    if state.get('clarification', {}).get('needs_improvement', False):
        message = input("Answer: ")
        if message.lower() == "exit":
            break
        state["messages"].append(HumanMessage(content=message))



🧩 Agent: clarificationAgent
📦 Output: {'clarification': {'needs_improvement': False, 'questions': []}}

🧩 Agent: keywordAgent
📦 Output: {'keywords': {'google_scholar_queries': ['preictal state recognition eeg', 'geometric deep learning seizure prediction', 'graph neural network epilepsy', 'spatio-temporal gcn eeg analysis', 'chb-mit scalp eeg dataset seizure onset', 'pre-seizure brain state detection wearable devices', 'early seizure detection gnn architecture', 'eeg-based seizure prediction using gcn', 'temporal pattern analysis epilepsy']}}

🧩 Agent: scholar_searcher
📦 Output: {'scholar_titles': {'Seizure prediction in scalp EEG using 3D convolutional neural networks with an image-based approach', 'A Meta-GNN approach to personalized seizure detection and classification', 'What is the present-day EEG evidence for a preictal state?', 'Classification of epileptic seizures in EEG data based on iterative gated graph convolution network', 'Dynamic gnns for precise seizure detection and c

TypeError: 'set' object is not subscriptable

In [ ]:
# async def semantic_scholar_searcher(state: AgentState) -> Optional[Set[str]]:
#     """
#     For each keyword:
#     - If open access PDF exists → download PDF.
#     - Else → save abstract as .txt file.
#     Returns the set of file paths saved.
#     """
#     keywords = state['keywords']['arxiv_phrases']
#     client = SemanticScholar()
#     saved_files: Set[str] = set()
#     os.makedirs("./papers", exist_ok=True)

#     for kw in keywords:
#         response = await asyncio.to_thread(
#             client.search_paper,
#             query=kw,
#             limit=5
#         )

#         for paper in response:
#             pdf_info = getattr(paper, 'openAccessPdf', None)
#             title = sanitize_filename(paper.title or paper.paperId)

#             if pdf_info and pdf_info.get('url'):
#                 pdf_path = f"./papers/{title}.pdf"
#                 await asyncio.to_thread(download_pdf, pdf_info['url'], pdf_path)
#                 saved_files.add(pdf_path)
#             else:
#                 abstract_text = paper.abstract or "[No abstract available]"
#                 txt_path = f"./papers/{title}.txt"
#                 save_abstract(abstract_text, txt_path)
#                 saved_files.add(txt_path)

#     return {'arxiv_papers':saved_files}

# def semantic_scholar_searcher(state: AgentState) -> Optional[Set[str]]:
#     """
#     For each keyword:
#     - If open access PDF exists → download PDF.
#     - Else → save abstract as .txt file.
#     Returns the set of file paths saved.
#     """
#     keywords = state['keywords']['arxiv_phrases']
#     client = SemanticScholar()
#     saved_files: Set[str] = set()
#     os.makedirs("./papers", exist_ok=True)

#     for kw in keywords:
#         response = client.search_paper(query=kw, limit=5)

#         for paper in itertools.islice(response, 5):
#             title = sanitize_filename(paper.title or paper.paperId)
#             doi = getattr(paper, "doi", None) or "NO_DOI"
#             journal = getattr(paper, "venue", None) or "SemanticScholar"

#             # File naming convention
#             txt_path = f"./papers/{title}@{doi}@{journal}.txt"
            
#             abstract_text = paper.abstract or "[No abstract available]"
#             txt_path = f"./papers/{title}.txt"
#             save_abstract(abstract_text, txt_path)
#             saved_files.add(txt_path)

#     return {'ss_papers': saved_files}


In [ ]:
# async def arxiv_searcher(state: AgentState) -> Optional[Set[str]]:
#     """
#     Return the URLs of 5 papers for each keyword from arxiv, and download PDFs.
#     """
#     results = set()
#     arxiv_phrases = state['keywords']['arxiv_phrases']

#     for phrase in arxiv_phrases:
#         search = arxiv.Search(query=phrase, max_results=5)
#         for paper in search.results():
#             results.add(paper.pdf_url)
            
#             safe_title = sanitize_filename(paper.title)
#             await asyncio.to_thread(
#                 paper.download_pdf,
#                 dirpath=f"./papers/{safe_title}"
#             )

#     return {'ss_papers':results}

# ssl_context = ssl.create_default_context(cafile=certifi.where())
# opener = urllib.request.build_opener(
#     urllib.request.HTTPSHandler(context=ssl_context)
# )
# urllib.request.install_opener(opener)

# def arxiv_searcher(state: dict) -> Optional[Set[str]]:
#     """
#     Return the titles (or DOIs if available) of 5 papers for each keyword from arxiv,
#     and save abstracts instead of PDFs.
#     """
#     results = set()
#     arxiv_phrases = state['keywords']['arxiv_phrases']

#     os.makedirs("./papers", exist_ok=True)

#     for phrase in arxiv_phrases:
#         search = arxiv.Search(query=phrase, max_results=5)
#         for paper in search.results():
#             # Gather metadata
#             safe_title = sanitize_filename(paper.title)
#             doi = paper.doi if paper.doi else "NO_DOI"
#             journal = paper.journal_ref if paper.journal_ref else "arXiv"

#             filename = f"./papers/{safe_title}@{doi}@{journal}.txt"

#             # Save abstract
#             with open(filename, "w", encoding="utf-8") as f:
#                 f.write(paper.summary.strip())

#             results.add(filename)

#     return {'ss_papers': results}
